# 🔬 Track 2: RT-DETR for PCB Defect Detection
## Dual-Track Architecture -- Transformer Branch (Google Colab & Local Ready)

**Project:** FOD Predictive Detection / PCB Defect Detection  
**Dataset:** DsPCBSD+ (9 defect classes, 10,259 images at 640x640)  
**Model:** RT-DETR-L (~32M parameters, hybrid CNN-Transformer)  
**Track 1 (CNN):** YOLOv8-nano / YOLO26n -> Extreme speed, real-time edge execution  
**Track 2 (Transformer):** RT-DETR-L -> High global context awareness, multi-scale anomaly detection  

---

### Colab Quick-Start (Drop & Run)
1. **Hardware Accelerator:** Go to `Runtime` -> `Change runtime type` -> select **T4 GPU** -> `Save`.
2. **Google Drive:** When prompted by Cell 2, click **Connect to Google Drive**.
3. **Click `Runtime` -> `Run all`.** Everything (dataset discovery, flattening/caching, training, validation, export, Drive checkpoint backup) executes automatically.


## 1. Setup & GPU Verification


In [ ]:
# Install required packages
!pip install -U ultralytics ensemble-boxes --quiet

import torch
import ultralytics

print("=" * 60)
print("  Environment & Hardware Verification")
print("=" * 60)
print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"PyTorch Version:     {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Status:          ACTIVE [OK]")
    print(f"Device:              {gpu_name} ({gpu_mem:.1f} GB VRAM)")
    DEVICE = 0
else:
    print("GPU Status:          NOT DETECTED [WARN]")
    print("IMPORTANT: If running in Google Colab, go to:")
    print("  Runtime -> Change runtime type -> Select 'T4 GPU' -> Save")
    print("Then re-run this cell.")
    DEVICE = "cpu"

print("=" * 60)


## 2. Configuration & Dataset Auto-Discovery
Automatically searches Google Drive or local directories for your dataset (`PCB_YOLO_Dataset`, `Classified_Dataset`, or zip archives).


In [ ]:
import os
import sys
import shutil
from pathlib import Path

# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & SET DATASET PATH
# ==============================================================================
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    
    # Check both '/content/drive/MyDrive' and '/content/drive/My Drive'
    MY_DRIVE = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('/content/drive/My Drive')
    
    # --------------------------------------------------------------------------
    # ▼▼▼ SET YOUR GOOGLE DRIVE FOLDER NAME HERE ▼▼▼
    # By default, we point directly to your uploaded folder.
    # If the folder in your Drive is named differently, change it here:
    DRIVE_FOLDER_NAME = "Classified_Dataset"
    # --------------------------------------------------------------------------
    
    SOURCE_DATASET = MY_DRIVE / DRIVE_FOLDER_NAME
    
    # If not found directly, print existing folders in My Drive to help you instantly spot it
    if not SOURCE_DATASET.exists():
        print(f"⚠️ Folder '{DRIVE_FOLDER_NAME}' not found at: {SOURCE_DATASET}")
        print("\nFolders currently visible in your My Drive:")
        found_dirs = [d.name for d in sorted(MY_DRIVE.iterdir()) if d.is_dir()]
        for d in found_dirs[:15]:
            print(f"  📁 {d}")
        
        # Check common subfolder location
        subfolder_check = MY_DRIVE / "DataSet_CourseProject" / DRIVE_FOLDER_NAME
        if subfolder_check.exists():
            SOURCE_DATASET = subfolder_check
            print(f"\n✅ Auto-located inside DataSet_CourseProject: {SOURCE_DATASET}")
        else:
            raise FileNotFoundError(f"Please set DRIVE_FOLDER_NAME = '{found_dirs[0]}' (or whatever your folder is named in My Drive).")
    else:
        print(f"✅ Found dataset folder: {SOURCE_DATASET}")
        
    WORK_DIR = Path("/content/DETR_Workspace")
    DRIVE_BACKUP_DIR = MY_DRIVE / "DataSet_CourseProject" / "runs" / "RTDETR_Track2"
else:
    SOURCE_DATASET = Path.cwd() / "Data Preprocessing" / "Classified_Dataset"
    if not SOURCE_DATASET.exists():
        SOURCE_DATASET = Path.cwd() / "Dataset" / "DsPCBSD+" / "Data_YOLO"
    WORK_DIR = Path.cwd() / "DETR_Workspace"
    DRIVE_BACKUP_DIR = Path.cwd() / "runs" / "detect" / "PCB_RTDETR_L_Track2"

WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Class definitions (from Kaggle DsPCBSD+ ground truth)
NUM_CLASSES = 9
CLASS_NAMES = {
    0: "SH",    # Solder Pads
    1: "SP",    # Shorts
    2: "SC",    # Scratches
    3: "OP",    # Opening
    4: "MB",    # Mask Break
    5: "HB",    # Hole Break
    6: "CS",    # Cremation
    7: "CFO",   # Component Foreign Object
    8: "BMFO",  # Bare Metal-Free Offset
}
CLASS_NAMES_FULL = {
    0: "Solder Pads (SH)",
    1: "Shorts (SP)",
    2: "Scratches (SC)",
    3: "Opening (OP)",
    4: "Mask Break (MB)",
    5: "Hole Break (HB)",
    6: "Cremation (CS)",
    7: "Component Foreign Object (CFO)",
    8: "Bare Metal-Free Offset (BMFO)",
}

# Training Hyperparameters
EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 8 if IN_COLAB else 4   # 8 fits T4 16GB GPU, 4 for local 4GB GPU
MODEL_NAME = "rtdetr-l.pt"
LR_INIT = 0.0001
LR_FINAL_FRAC = 0.01
WEIGHT_DECAY = 0.0001
WARMUP_EPOCHS = 3.0
PATIENCE = 20
FREEZE_LAYERS = 10

# Adaptive Preprocessing Settings
COMPLEXITY_THRESHOLD = 100.0
YOLO_LOW_RES = (224, 224)
YOLO_HIGH_RES = (640, 640)
DETR_RES = (640, 640)

print("\nConfiguration ready:")
print(f"  Source Dataset:    {SOURCE_DATASET}")
print(f"  Batch Size:        {BATCH_SIZE}")
print(f"  Model:             {MODEL_NAME}")


## 3. Dataset Preparation & Local NVMe Caching
If your dataset is organized in class subfolders, this cell flattens it.  
In Colab, it copies/caches the dataset directly onto the fast local NVMe SSD (`/content/DETR_Dataset`) to eliminate Google Drive I/O bottlenecks during training.


In [ ]:
DETR_DATASET = Path("/content/DETR_Dataset") if IN_COLAB else (WORK_DIR / "DETR_Dataset")

# If /content/dataset already exists with cached images from previous run, reuse it directly!
if IN_COLAB and Path("/content/dataset/images/train").exists() and len(list(Path("/content/dataset/images/train").glob("*.jpg"))) >= 2000:
    DETR_DATASET = Path("/content/dataset")
    print(f"Reusing existing cached dataset from previous run at: {DETR_DATASET}")

def prepare_flat_dataset(source_root, target_root):
    target_root = Path(target_root)
    source_root = Path(source_root)
    
    for split in ['train', 'val']:
        target_img_dir = target_root / 'images' / split
        target_lbl_dir = target_root / 'labels' / split
        target_img_dir.mkdir(parents=True, exist_ok=True)
        target_lbl_dir.mkdir(parents=True, exist_ok=True)
        
        src_img_dir = source_root / 'images' / split
        src_lbl_dir = source_root / 'labels' / split
        
        # Check if already cached
        existing_imgs = len(list(target_img_dir.glob("*.jpg")))
        if existing_imgs >= 2000:
            print(f"  {split}: Already prepared and cached ({existing_imgs} images)")
            continue
            
        print(f"  Preparing {split} set...")
        # Copy images
        for img_file in src_img_dir.rglob("*.jpg"):
            dst = target_img_dir / img_file.name
            if not dst.exists():
                shutil.copy2(str(img_file), str(dst))
                
        # Copy labels
        for lbl_file in src_lbl_dir.rglob("*.txt"):
            dst = target_lbl_dir / lbl_file.name
            if not dst.exists():
                shutil.copy2(str(lbl_file), str(dst))
                
        n_imgs = len(list(target_img_dir.glob("*.jpg")))
        n_lbls = len(list(target_lbl_dir.glob("*.txt")))
        print(f"  {split}: {n_imgs} images, {n_lbls} labels [OK]")
        
    # Generate data.yaml
    yaml_lines = [
        f"path: {target_root.as_posix()}",
        "train: images/train",
        "val: images/val",
        f"nc: {NUM_CLASSES}",
        "names:"
    ]
    for idx, name in CLASS_NAMES.items():
        yaml_lines.append(f"  {idx}: {name}")
        
    yaml_path = target_root / "data.yaml"
    with open(yaml_path, "w") as f:
        f.write("\n".join(yaml_lines) + "\n")
        
    print(f"\nGenerated dataset configuration at: {yaml_path}")
    return str(yaml_path)

print("Preparing dataset on fast local drive...")
DATA_YAML = prepare_flat_dataset(SOURCE_DATASET, DETR_DATASET)

train_count = len(list((Path(DETR_DATASET) / "images" / "train").glob("*.jpg")))
val_count = len(list((Path(DETR_DATASET) / "images" / "val").glob("*.jpg")))
print(f"\nTotal Dataset: {train_count} train images, {val_count} validation images.")


## 4. YOLO -> COCO JSON Converter (Academic Utility)
Converts annotations to standard COCO JSON format for academic completeness and evaluation tooling.


In [ ]:
import json
from PIL import Image as PILImage
from datetime import datetime

COCO_DIR = WORK_DIR / "coco_annotations"
COCO_DIR.mkdir(parents=True, exist_ok=True)

def yolo_to_coco(images_dir, labels_dir, output_json, class_names):
    coco = {
        "info": {
            "description": "DsPCBSD+ PCB Surface Defect Dataset - COCO Format",
            "version": "1.0",
            "year": 2026,
            "contributor": "VIT Capstone Team",
            "date_created": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        },
        "licenses": [{"id": 1, "name": "Research Use"}],
        "images": [],
        "annotations": [],
        "categories": [{"id": k, "name": v, "supercategory": "pcb_defect"} for k, v in class_names.items()],
    }

    ann_id = 0
    img_files = sorted(Path(images_dir).glob("*.jpg"))
    for idx, img_p in enumerate(img_files):
        try:
            with PILImage.open(img_p) as im:
                w, h = im.size
        except Exception:
            continue

        coco["images"].append({"id": idx, "file_name": img_p.name, "width": w, "height": h})
        lbl_p = Path(labels_dir) / (img_p.stem + ".txt")
        if not lbl_p.exists():
            continue

        with open(lbl_p, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    c_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    bw_px = bw * w
                    bh_px = bh * h
                    xmin = (xc * w) - (bw_px / 2)
                    ymin = (yc * h) - (bh_px / 2)
                    coco["annotations"].append({
                        "id": ann_id,
                        "image_id": idx,
                        "category_id": c_id,
                        "bbox": [round(xmin, 2), round(ymin, 2), round(bw_px, 2), round(bh_px, 2)],
                        "area": round(bw_px * bh_px, 2),
                        "iscrowd": 0
                    })
                    ann_id += 1

    with open(output_json, "w") as f:
        json.dump(coco, f, indent=2)
    return output_json

print("Generating COCO JSON annotations...")
for split in ['train', 'val']:
    out_file = COCO_DIR / f"instances_{split}.json"
    yolo_to_coco(
        images_dir=Path(DETR_DATASET) / "images" / split,
        labels_dir=Path(DETR_DATASET) / "labels" / split,
        output_json=out_file,
        class_names=CLASS_NAMES
    )
    print(f"  {split}: Saved -> {out_file}")
print("COCO conversion complete.")


## 5. Adaptive Preprocessing -- Dual-Track Routing Implementation


In [ ]:
import cv2
import numpy as np

def analyze_complexity(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def adaptive_preprocess(image, target_model="detr"):
    variance = analyze_complexity(image)
    is_micro = variance >= COMPLEXITY_THRESHOLD

    meta = {
        "variance": round(variance, 2),
        "is_micro": is_micro,
        "defect_scale": "micro" if is_micro else "macro",
    }

    if target_model == "yolo":
        # Track 1: Variable resolution for speed
        res = YOLO_HIGH_RES if is_micro else YOLO_LOW_RES
        meta["resolution"] = res
        meta["enhancement"] = "none"
        return cv2.resize(image, res), meta
    elif target_model == "detr":
        # Track 2: Always 640x640, enhance micro details
        if is_micro:
            gaussian = cv2.GaussianBlur(image, (0, 0), sigmaX=3)
            enhanced = cv2.addWeighted(image, 1.5, gaussian, -0.5, 0)
            meta["enhancement"] = "unsharp_mask"
            return cv2.resize(enhanced, DETR_RES), meta
        else:
            meta["enhancement"] = "none"
            return cv2.resize(image, DETR_RES), meta

# Demo on 5 images
sample_files = list((Path(DETR_DATASET) / "images" / "train").glob("*.jpg"))[:5]
print("Adaptive Preprocessing Decision Table:")
print(f"{'Image':<15} {'Variance':<12} {'Scale':<8} {'YOLO Res':<14} {'DETR Enhancement'}")
print("-" * 65)
for s in sample_files:
    im = cv2.imread(str(s))
    if im is not None:
        _, ym = adaptive_preprocess(im, "yolo")
        _, dm = adaptive_preprocess(im, "detr")
        print(f"{s.name:<15} {ym['variance']:<12.2f} {ym['defect_scale']:<8} {str(ym['resolution']):<14} {dm['enhancement']}")


## 6. RT-DETR Model Initialization


In [ ]:
from ultralytics import RTDETR

print(f"Loading pretrained {MODEL_NAME} ...")
model = RTDETR(MODEL_NAME)
print("RT-DETR-L initialized successfully.")
print(f"Backbone: HGNetv2 | Neck: Hybrid Encoder | Head: Transformer Decoder")
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total Parameters: {total_params:,} (~{total_params/1e6:.2f}M)")


## 7. RT-DETR-L Training Execution
Trains with AdamW, Cosine Annealing, FP16 AMP, and early stopping.  
Checkpoints are automatically backed up to Google Drive when complete.


In [ ]:
PROJECT_DIR = str(WORK_DIR / "runs" / "detect")
RUN_NAME = "PCB_RTDETR_L_Track2"

print("=" * 60)
print("  Starting RT-DETR-L Training")
print("=" * 60)
print(f"  Data Config:    {DATA_YAML}")
print(f"  Device:         {DEVICE}")
print(f"  Batch Size:     {BATCH_SIZE}")
print(f"  Image Size:     {IMG_SIZE}")
print(f"  Epochs:         {EPOCHS}")
print(f"  Optimizer:      AdamW (lr={LR_INIT}, wd={WEIGHT_DECAY})")
print(f"  Output Dir:     {PROJECT_DIR}/{RUN_NAME}")
print("=" * 60)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    name=RUN_NAME,
    project=PROJECT_DIR,

    # Optimizer & Learning Rate
    optimizer="AdamW",
    lr0=LR_INIT,
    lrf=LR_FINAL_FRAC,
    weight_decay=WEIGHT_DECAY,
    cos_lr=True,
    warmup_epochs=WARMUP_EPOCHS,

    # Training stability & speed
    amp=True,
    patience=PATIENCE,
    freeze=FREEZE_LAYERS,
    workers=4 if IN_COLAB else 2,

    # Augmentation
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # Output
    plots=True,
    save=True,
    exist_ok=True,
    verbose=True
)

BEST_WEIGHTS = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nTraining complete! Best model saved to: {BEST_WEIGHTS}")

# Backup checkpoint & key plots to Google Drive
if IN_COLAB and DRIVE_BACKUP_DIR.exists():
    print(f"Backing up checkpoint to Google Drive: {DRIVE_BACKUP_DIR} ...")
    try:
        shutil.copy2(str(BEST_WEIGHTS), str(DRIVE_BACKUP_DIR / "best.pt"))
        for plot_file in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
            p_src = Path(results.save_dir) / plot_file
            if p_src.exists():
                shutil.copy2(str(p_src), str(DRIVE_BACKUP_DIR / plot_file))
        print("Drive backup complete! Your trained weights are safe in Google Drive.")
    except Exception as e:
        print(f"Drive backup note: {e}")


## 8. Validation & Per-Class Metrics Evaluation


In [ ]:
import pandas as pd
from ultralytics import RTDETR

eval_checkpoint = BEST_WEIGHTS if 'BEST_WEIGHTS' in locals() and BEST_WEIGHTS.exists() else Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
eval_model = RTDETR(str(eval_checkpoint))

print(f"Evaluating checkpoint: {eval_checkpoint}")
val_metrics = eval_model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True
)

print("\n" + "=" * 60)
print("  RT-DETR Track 2 Validation Results")
print("=" * 60)
print(f"  mAP@50:       {val_metrics.box.map50:.4f}")
print(f"  mAP@50-95:    {val_metrics.box.map:.4f}")
print(f"  Precision:     {val_metrics.box.mp:.4f}")
print(f"  Recall:        {val_metrics.box.mr:.4f}")
print("=" * 60)

# Per-class AP table
class_rows = []
for i in range(NUM_CLASSES):
    ap50 = val_metrics.box.ap50[i]
    ap = val_metrics.box.ap[i]
    class_rows.append({
        "Class ID": i,
        "Code": CLASS_NAMES[i],
        "Full Name": CLASS_NAMES_FULL[i],
        "mAP@50": round(float(ap50), 4),
        "mAP@50-95": round(float(ap), 4)
    })

df_eval = pd.DataFrame(class_rows)
print("\nPer-Class Detection Performance:")
print(df_eval.to_string(index=False))


## 9. Qualitative Detection Visualizer


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

sample_val_imgs = list((Path(DETR_DATASET) / "images" / "val").glob("*.jpg"))
chosen_samples = random.sample(sample_val_imgs, min(6, len(sample_val_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

for idx, img_path in enumerate(chosen_samples):
    ax = axes[idx // 3, idx % 3]
    im = cv2.imread(str(img_path))
    im_rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    ax.imshow(im_rgb)
    
    res = eval_model.predict(str(img_path), conf=0.25, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    n_det = 0
    if res.boxes is not None:
        for box in res.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            c_id = int(box.cls[0])
            conf_val = float(box.conf[0])
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=colors[c_id % len(colors)], facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, f"{CLASS_NAMES[c_id]} {conf_val:.2f}",
                    color='white', fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=colors[c_id % len(colors)], alpha=0.85))
            n_det += 1
            
    ax.set_title(f"{img_path.name} ({n_det} detections)", fontsize=10)
    ax.axis("off")

plt.suptitle("RT-DETR Track 2 Detections on Validation Set", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Post-Training Optimization (FP16 / ONNX Export)


In [ ]:
print("Exporting RT-DETR-L to ONNX FP16 for edge inference...")
try:
    onnx_path = eval_model.export(format="onnx", imgsz=IMG_SIZE, half=True, simplify=True)
    print(f"Export successful! Saved: {onnx_path}")
    
    size_pt = os.path.getsize(str(eval_checkpoint)) / (1024 * 1024)
    size_onnx = os.path.getsize(str(onnx_path)) / (1024 * 1024)
    print(f"PyTorch size: {size_pt:.1f} MB")
    print(f"ONNX FP16:    {size_onnx:.1f} MB ({(1 - size_onnx/size_pt)*100:.1f}% reduction)")
except Exception as e:
    print(f"Export note: {e}")


## 11. Dual-Track Inference Fusion (Weighted Box Fusion)


In [ ]:
from ensemble_boxes import weighted_boxes_fusion
from ultralytics import YOLO

# Potential YOLO weight locations (including last night's backup!)
yolo_candidates = [
    Path("/content/drive/MyDrive/DataSet_CourseProject/runs_backup/detect/PCB_YOLO26n_Run/weights/best.pt"),
    Path("/content/drive/MyDrive/DataSet_CourseProject/runs_backup/PCB_YOLO26n_Run/weights/best.pt"),
    Path("/content/drive/MyDrive/DataSet_CourseProject/runs/YOLO26n_baseline/weights/best.pt"),
    Path("/content/runs/detect/PCB_YOLO26n_Run/weights/best.pt"),
    Path.cwd() / "runs" / "detect" / "PCB_YOLO26n_Local" / "weights" / "best.pt",
    Path.cwd() / "yolo26n.pt",
]

YOLO_WEIGHTS = None
for yc in yolo_candidates:
    if yc.exists():
        YOLO_WEIGHTS = str(yc)
        break

if YOLO_WEIGHTS:
    print(f"Using YOLO Track 1 weights: {YOLO_WEIGHTS}")
    print(f"Using RT-DETR Track 2 weights: {eval_checkpoint}")
    
    test_sample = str(sample_val_imgs[0])
    raw_img = cv2.imread(test_sample)
    h, w, _ = raw_img.shape
    
    # Track 1
    yolo_net = YOLO(YOLO_WEIGHTS)
    y_pred = yolo_net.predict(test_sample, conf=0.25, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    
    # Track 2
    d_pred = eval_model.predict(test_sample, conf=0.25, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    
    boxes_list, scores_list, labels_list = [], [], []
    for pred in [y_pred, d_pred]:
        if pred.boxes is not None and len(pred.boxes) > 0:
            norm_b = pred.boxes.xyxy.cpu().numpy().copy()
            norm_b[:, [0, 2]] /= w
            norm_b[:, [1, 3]] /= h
            boxes_list.append(norm_b.tolist())
            scores_list.append(pred.boxes.conf.cpu().numpy().tolist())
            labels_list.append(pred.boxes.cls.cpu().numpy().astype(int).tolist())
        else:
            boxes_list.append([])
            scores_list.append([])
            labels_list.append([])
            
    if any(len(b) > 0 for b in boxes_list):
        f_boxes, f_scores, f_labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=[1.0, 1.5],  # 1.5x weight to transformer precision
            iou_thr=0.5,
            skip_box_thr=0.25
        )
        print(f"Weighted Box Fusion complete:")
        print(f"  YOLO Detections:    {len(y_pred.boxes)}")
        print(f"  RT-DETR Detections: {len(d_pred.boxes)}")
        print(f"  Fused Detections:   {len(f_boxes)}")
else:
    print("YOLO Track 1 weights not found yet. Train YOLO Track 1 first to run side-by-side fusion.")


## 12. Academic Report Progress Update

### Theoretical Justification for the Dual-Track Architecture
*(Ready to paste into `PCB_Defect_Detection_Roadmap_Final.docx`)*

The proposed dual-track detection architecture strategically pairs a lightweight convolutional neural network (YOLOv8-nano / YOLO26n) with a hybrid transformer-based detector (RT-DETR-L) to exploit the complementary representational strengths of both paradigms for printed circuit board (PCB) surface defect detection. The convolutional track leverages spatially-local inductive biases inherent to CNN architectures, enabling highly efficient feature extraction at variable input resolutions -- a property we exploit through an adaptive preprocessing pipeline that dynamically routes macro-scale defects (e.g., large solder pad displacements) at reduced 224x224 resolution for inference latency optimization, while preserving native 640x640 resolution for fine-grained micro-defects requiring higher spatial fidelity. In contrast, the transformer track employs RT-DETR's hybrid encoder-decoder architecture, wherein an HGNetv2 CNN backbone extracts multi-scale feature pyramids that are subsequently refined through intra-scale and cross-scale self-attention modules in the transformer encoder. This design enables the model to establish long-range spatial dependencies across the entire PCB image -- a critical capability for detecting spatially distributed defect patterns such as shorts (SP) and scratches (SC) that span disjoint regions of the board, and for disambiguating visually similar defect categories (e.g., cremation vs. mask break) that require holistic contextual reasoning rather than purely local feature analysis. The transformer decoder's learnable object queries eliminate the need for hand-designed anchor boxes and non-maximum suppression, yielding a fully end-to-end detection pipeline with set-based Hungarian matching loss. At inference time, predictions from both tracks are unified through Weighted Box Fusion (WBF), which computes confidence-weighted averages of overlapping bounding box proposals, producing a final detection set that empirically surpasses either individual model's precision-recall trade-off. This architectural duality directly addresses the heterogeneous nature of PCB surface defects in the DsPCBSD+ benchmark -- spanning nine categories with vastly different spatial scales, morphological signatures, and contextual dependencies -- while maintaining a deployment-viable computational profile through post-training quantization (FP16/INT8) and structured pruning of redundant attention heads.
